In [1]:
import pandas as pd
import numpy as np
import json
import re, pytz, os, requests, sys
from pathlib import Path
from datetime import datetime
import sys
sys.path.append("/workspaces/service-data")

from src.clean import clean_percentage, clean_fiscal_yr, normalize_string, standardize_column_names
from src.load import load_csv, download_csv_files
from src.export import export_to_csv
from src.merge import merge_si, merge_ss
from src.utils import dept_list, program_list
from main import get_config

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

base_dir = Path.cwd()
parent_dir = base_dir.parent
config = get_config()
# dept = dept_list(config)


In [7]:
ib_orgs = load_csv('infobase_orgs.csv', config)

# Restrict list of orgs to those active after the year 2000. Tidier dataset.
ib_orgs = ib_orgs.loc[ib_orgs['year']>=2000]

# Remove all but the latest year's worth of data for each org.
ib_orgs = pd.merge(
    ib_orgs.groupby(['id'], as_index=False)['year'].max(),
    ib_orgs[['year', 'id', 'legal_title_en', 'applied_title_en', 'legal_title_fr', 'applied_title_fr']],
    on=['id', 'year']
    )

# Define the preferred org name as the applied title, default to legal title if it doesn't exist.
ib_orgs['department_en'] = ib_orgs.loc[:, 'applied_title_en'].fillna(ib_orgs.loc[:, 'legal_title_en'])
ib_orgs['department_fr'] = ib_orgs.loc[:, 'applied_title_fr'].fillna(ib_orgs.loc[:, 'legal_title_fr'])

ib_orgs=(ib_orgs
            .rename(columns={'id':'org_id'})
            .drop(columns=['year', 'legal_title_en', 'applied_title_en', 'legal_title_fr', 'applied_title_fr'])
            .drop_duplicates()
)

# Define org type as a string rather than an integer
ib_orgs['org_id'] = ib_orgs['org_id'].astype(str)


# Standardize column names and convert org_id to string
dept = standardize_column_names(ib_orgs)


In [ ]:
def dept_list(config, export=False):
    """
    Get a list of departments with their English and French names.
    """
    try:
        ib_orgs = load_csv('infobase_orgs.csv', config)

        # Restrict list of orgs to those active after the year 2000. Tidier dataset.
        ib_orgs = ib_orgs.loc[ib_orgs['year']>=2000]

        # Remove all but the latest year's worth of data for each org.
        ib_orgs = pd.merge(
            ib_orgs.groupby(['id'], as_index=False)['year'].max(),
            ib_orgs[['year', 'id', 'legal_title_en', 'applied_title_en', 'legal_title_fr', 'applied_title_fr']],
            on=['id', 'year']
            )

        # Define the preferred org name as the applied title, default to legal title if it doesn't exist.
        ib_orgs['department_en'] = ib_orgs.loc[:, 'applied_title_en'].fillna(ib_orgs.loc[:, 'legal_title_en'])
        ib_orgs['department_fr'] = ib_orgs.loc[:, 'applied_title_fr'].fillna(ib_orgs.loc[:, 'legal_title_fr'])

        ib_orgs=(ib_orgs
                    .rename(columns={'id':'org_id'})
                    .drop(columns=['year', 'legal_title_en', 'applied_title_en', 'legal_title_fr', 'applied_title_fr'])
                    .drop_duplicates()
        )

        # Define org type as a string rather than an integer
        ib_orgs['org_id'] = ib_orgs['org_id'].astype(str)

        
        # Standardize column names and convert org_id to string
        dept = standardize_column_names(ib_orgs)
        
        if export:
            UTILS_DIR = config['output_dir'] / config['utils_dir']
            export_to_csv(
                data_dict={'dept': dept},
                output_dir=UTILS_DIR,
            )
        
        return dept
    
    except Exception as e:
        logger.error("Error: %s", e, exc_info=True)